# 03 — ML training and reconstruction dashboard

This notebook is the heart of CALOMAPS:

1. **Train** (or load) a Deep Quantile Ensemble surrogate model for each of the 4 readouts (Analog, MIP, Hits, Cluster).
2. **Reconstruct** energy via Neyman inversion of the surrogate.
3. **Plot** the 3-panel physics dashboard.

You need a GPU for training. See [`docs/handbook.md`](../docs/handbook.md) §11.2 for the CUDA torch setup recipe. Training is part of the workflow — there are no pretrained ensembles in the repo; the training cell writes per-particle ensembles into `models/saved_ensembles_gpu_<tag>/`.

**Kernel**: `Key4hep + GPU`. Verify GPU availability in the very first cell.


---

### How to use this notebook

This is the **scaffold**. The science cells are blanked out with a `# TODO` block (task / hint / expected result) and a `raise NotImplementedError`, so an unfilled cell fails loudly.

**One particle at a time.** Set `PARTICLE` in the cell below — `"gamma"` (photons) or `"pi+"` (pions). Everything else is derived from that, so the *same* cells run on a *different* dataset; run both and **compare the dashboards at the end** — the photon-vs-pion resolution contrast is the headline result.

**The capstone** is the last section: replace the naive clustering and *measure* the payoff on the dashboard.

## 1. Verify the kernel + GPU

In [ ]:
import torch, sys
print(f"torch version: {torch.__version__}")
print(f"torch path:    {torch.__file__}")
print(f"cuda built:    {torch.backends.cuda.is_built()}")
print(f"cuda available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print()
    print("WARNING: no CUDA. You're on a CPU-only torch — training will take ~30 min instead of ~9 min.")
    print("To enable GPU: in a terminal run  bash $CALOMAPS_HOME/setup/setup_gpu_kernel.sh,")
    print("then reopen this notebook on the 'Key4hep + GPU' kernel (see handbook §11.2).")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")


## 1b. Pick your particle

This single cell is the *only* thing that differs per particle. It selects the particle, the dataset directory, the per-particle `.npz` that nb02 wrote, and a per-particle ensemble output directory — so the trained models for photons and pions never overwrite each other.

- photons: leave `PARTICLE = "gamma"`
- pions: set `PARTICLE = "pi+"`

In [ ]:
# === DATASET SELECTION — the one cell that differs per particle ===
PARTICLE = "gamma"   # "gamma" or "pi+"

# Everything below is DERIVED from PARTICLE — do not edit per particle.
DATASET = ("data_spectrum_100um_400GeV" if PARTICLE == "gamma"
           else "data_spectrum_100um_400GeV_piplus")
# nb02 writes one .npz per particle (decal_extracted_data_<tag>.npz, tag = gamma|piplus);
# nb03 reads the matching one — names MUST agree with nb02's OUT_NAME or the load fails.
PART_TAG = "gamma" if PARTICLE == "gamma" else "piplus"
NPZ_NAME = f"decal_extracted_data_{PART_TAG}.npz"
# per-particle ensemble dir so photon/pion checkpoints don't collide
ENS_SUBDIR = ("saved_ensembles_gpu_gamma" if PARTICLE == "gamma"
              else "saved_ensembles_gpu_piplus")
# (the raw-file glob nb01/nb02 use, kept here for reference / sanity)
FILE_GLOB = ("sim_photons_part*.root" if PARTICLE == "gamma"
             else "sim_piplus_part*.root")

print(f"PARTICLE   = {PARTICLE}")
print(f"DATASET    = {DATASET}")
print(f"NPZ_NAME   = {NPZ_NAME}")
print(f"ENS_SUBDIR = {ENS_SUBDIR}")


## 2. Load the extracted dataset

In [ ]:
import os, sys, numpy as np
# Resolve the repo root. Prefer $CALOMAPS_HOME (exported by `source setup_calomaps.sh`
# in a terminal); otherwise auto-locate it by walking up from the kernel's working
# directory, so the notebook also works when opened from the JupyterLab GUI — whose
# kernel does NOT inherit a terminal's environment.
def _calomaps_home():
    h = os.environ.get("CALOMAPS_HOME")
    if h and os.path.isdir(os.path.join(h, "geometry")):
        return h
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.isdir(os.path.join(p, "geometry")) and os.path.isdir(os.path.join(p, "sim")):
            return p
        p = os.path.dirname(p)
    return os.path.expanduser("~/CALOMAPS")
CALOMAPS_HOME = _calomaps_home()

# Make our analysis utilities importable. These modules are GIVEN machinery —
# you call them, you don't have to reimplement them.
sys.path.insert(0, os.path.join(CALOMAPS_HOME, "analysis"))
from quantilenet import QuantileNet, quantile_loss, save_ensemble, load_ensemble, QUANTILES
from dashboard import get_interpolators, reco_metrics_over_grid, plot_dashboard

# Read the per-particle .npz that YOU produced in nb02 (NPZ_NAME from the
# selection cell). If this file is missing, go back and run nb02 for your particle.
npz_path = os.path.join(CALOMAPS_HOME, "models", NPZ_NAME)
data = np.load(npz_path)
all_truth   = data["all_truth"]
all_visible = data["all_visible"]
all_mip     = data["all_mip"]
all_hits    = data["all_hits"]
all_cluster = data["all_cluster"]
print(f"loaded {len(all_truth)} events from {npz_path}")

valid = (all_hits > 0) & (all_truth > 0) & (all_visible > 0) & (all_mip > 0) & (all_cluster > 0)
x_train = all_truth[valid]
print(f"valid events for training: {valid.sum()}")


## 3. Train the ensembles

`RETRAIN = True` trains one Deep Quantile Ensemble per readout into
`$CALOMAPS_HOME/models/saved_ensembles_gpu_<tag>/` (~10 min on the A100). On a
later re-run, set `RETRAIN = False` to reload the ensembles you already trained.


In [ ]:
from quantilenet import train_one_ensemble

# RETRAIN=True: you train your OWN per-particle ensembles.
RETRAIN = True
ENSEMBLE_DIR = os.path.join(CALOMAPS_HOME, "models", ENS_SUBDIR)
os.makedirs(ENSEMBLE_DIR, exist_ok=True)

# TODO (your task): train (or, with RETRAIN=False, load) one Deep Quantile
#   Ensemble per readout, then load all four back into ens_a/ens_m/ens_h/ens_c.
#   The four readouts are the four arrays from nb02:
#       all_visible -> 'True Analog'      -> ens_analog.pth
#       all_mip     -> 'MIP Proxy'        -> ens_mip.pth
#       all_hits    -> 'Raw Hits'         -> ens_hits.pth
#       all_cluster -> 'Naive Clustering' -> ens_cluster.pth
# Hint: train_one_ensemble(x_train, y_arr[valid], device, name=..., seed_base=...)
#   returns (ens, xmax, ymax). Persist it with
#   save_ensemble(ens, xmax, ymax, os.path.join(ENSEMBLE_DIR, fname)).
#   Use a DIFFERENT seed_base per readout (e.g. 1000/2000/3000/4000) so the
#   bootstrap splits differ. Then load each back with
#   load_ensemble(path, device) -> (ens, x_max, y_frac_max).
# Expected: 4 ensembles, ~20 models each, on device=cuda. Training all four
#   takes ~10 min on the A100. Loading should print '... 20 models each'.
# Solution: not distributed with this repository.
raise NotImplementedError("fill this in")

# After your code above, these four must exist for the rest of the notebook:
#   ens_a, xa, ya = load_ensemble(os.path.join(ENSEMBLE_DIR, "ens_analog.pth"),  device)
#   ens_m, xm, ym = load_ensemble(os.path.join(ENSEMBLE_DIR, "ens_mip.pth"),     device)
#   ens_h, xh, yh = load_ensemble(os.path.join(ENSEMBLE_DIR, "ens_hits.pth"),    device)
#   ens_c, xc, yc = load_ensemble(os.path.join(ENSEMBLE_DIR, "ens_cluster.pth"), device)


## 4. The raw data the surrogate learns from

Before any model, look at what we are modeling. Each event gives a true beam
energy `E_true` and four *readouts* — different ways the DECAL could report that
energy:

- **True Analog** — total energy deposited in silicon (the ideal analog sum).
- **MIP Proxy** — each hit weighted by `E_hit / E_MIP` (what a MIP-counting chip approximates).
- **Raw Hits** — just the number of lit pixels (the pure digital readout).
- **Naive 2D Clustering** — the count of 8-connected pixel clusters, summed over all layers.

Every readout is a **noisy, possibly non-linear function of `E_true`**. The
scatter below is the entire training set the surrogates see. Each ensemble's job
is to learn, for a given `E_true`, the *distribution* of its readout — the
typical value and the spread.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# GIVEN: the readout dictionary + a consistent colour per readout. Reuse these
# names/colours in every plot below so your panels stay comparable.
readouts = {
    "True Analog":          all_visible[valid],
    "MIP Proxy":            all_mip[valid],
    "Raw Hits":             all_hits[valid],
    "Naive 2D Clustering":  all_cluster[valid],
}
colors = {"True Analog": "royalblue", "MIP Proxy": "forestgreen",
          "Raw Hits": "crimson", "Naive 2D Clustering": "darkorchid"}

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (name, y) in zip(axes.flat, readouts.items()):
    # TODO (your task): scatter this readout (y) against E_true (x_train) on ax.
    # Hint: a single ax.scatter of the readout (y) vs x_train -- small,
    #   semi-transparent markers (so density shows) and color=colors[name].
    # Expected: 4 noisy point-clouds rising with E_true. Note which look tight and
    #   which look noisy -- you will quantify this on the dashboard.
    # Solution: not distributed with this repository.
    raise NotImplementedError("fill this in")
    ax.set_title(f"{name}  vs  $E_{{true}}$")
    ax.set_xlabel("$E_{true}$ [GeV]")
    ax.set_ylabel(f"{name} readout")
    ax.grid(True, alpha=0.3)
fig.suptitle("Raw readout vs true energy — the data each ensemble is trained on", fontsize=13)
plt.tight_layout()
plt.show()


## 5. What each ensemble learned

Each ensemble is 20 tiny networks. Instead of predicting a single response
value, every network predicts **three quantiles** of the response at a given
`E_true`: the 15.87%, 50%, and 84.13% points — the median and the symmetric ±1σ
envelope of a Gaussian (no Gaussianity is assumed; the
[pinball loss](../analysis/quantilenet.py) simply drives each output to its
target percentile).

So the learned object is not a curve but a **band**:

- the **median** curve is the calibration (typical response vs energy),
- the **half-width** of the band is the intrinsic resolution σ(E) of that readout
  *before* reconstruction.

Overlaying the band on the raw points shows how well 3000 parameters capture the
response.

In [ ]:
def ensemble_curves(ens, x_max, y_frac_max, e_grid):
    # GIVEN machinery: ensemble-averaged absolute (low, med, high) readout
    # quantiles vs E. Returns three arrays (lo, med, hi) over e_grid.
    xt = torch.tensor(e_grid / x_max, dtype=torch.float32, device=device).unsqueeze(1)
    preds = []
    for m in ens:
        m.eval()
        with torch.no_grad():
            preds.append(m(xt).cpu().numpy())
    avg = np.mean(preds, axis=0) * y_frac_max * e_grid[:, None]
    return avg[:, 0], avg[:, 1], avg[:, 2]

e_grid = np.linspace(5, 400, 400)
ensembles = {
    "True Analog":          (ens_a, xa, ya),
    "MIP Proxy":            (ens_m, xm, ym),
    "Raw Hits":             (ens_h, xh, yh),
    "Naive 2D Clustering":  (ens_c, xc, yc),
}

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (name, y) in zip(axes.flat, readouts.items()):
    # TODO (your task): get the learned (lo, med, hi) band for this readout with
    #   ensemble_curves(*ensembles[name], e_grid), then overlay it on the raw points.
    # Hint: ensemble_curves(*ensembles[name], e_grid) returns (lo, med, hi) over
    #   e_grid. Overlay them on the raw scatter -- the median as a line, the lo-hi
    #   range as a shaded band (fill_between).
    # Expected: a black median curve threading the point-cloud with a grey ±1σ band
    #   hugging it. The band is the readout's intrinsic resolution BEFORE inversion.
    # Solution: not distributed with this repository.
    raise NotImplementedError("fill this in")
    ax.set_title(f"{name}")
    ax.set_xlabel("$E_{true}$ [GeV]")
    ax.set_ylabel(f"{name} readout")
    ax.legend(loc="upper left", fontsize=9)
    ax.grid(True, alpha=0.3)
fig.suptitle(r"Learned quantile surrogate (median + $\pm1\sigma$ band) over the raw data", fontsize=13)
plt.tight_layout()
plt.show()


### Spread across the ensemble

The band above is the *learned* ±1σ of the response. Separately, the **20 networks
disagree** with each other (they each saw a different bootstrap split) — that
disagreement is the **epistemic** uncertainty. Plotting every member's median *and ±1σ quantile* curves shows how tightly the
ensemble has pinned down both the response and its width.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
e_g = np.linspace(5, 400, 200)
for ax, (name, y) in zip(axes.flat, readouts.items()):
    ens, x_max, y_frac_max = ensembles[name]
    ax.scatter(x_train, y, s=3, alpha=0.06, color="gray")
    # TODO (your task): loop over the individual networks in `ens` and draw EACH
    #   member's median (col 1) in the readout colour and its ±1σ quantiles
    #   (cols 0 and 2) in faint black. The scatter of the curves IS the epistemic
    #   (model) uncertainty.
    # Hint: for each model in `ens`, evaluate it on the energy grid (eval mode,
    #   no_grad) and un-normalise like ensemble_curves does (* y_frac_max * e_g).
    #   Plot every member's median curve in the readout colour; the spread of those
    #   curves is the epistemic (model) uncertainty.
    # Expected: ~20 nearly-overlapping median curves (tight = ensemble agrees) with
    #   a thin black ±1σ halo. Wider spread for pions / for the noisier readouts.
    # Solution: not distributed with this repository.
    raise NotImplementedError("fill this in")
    ax.set_title(name); ax.set_xlabel("$E_{true}$ [GeV]"); ax.set_ylabel(f"{name} readout")
    ax.legend(loc="upper left", fontsize=9); ax.grid(True, alpha=0.3)
fig.suptitle("Ensemble spread: every network's median (colour) and ±1σ quantiles (black) — epistemic uncertainty", fontsize=12)
plt.tight_layout()
plt.show()


## 6. Neyman inversion — from a measurement to an energy

The surrogate runs *forward*: `E_true → response`. Reconstruction is the
*inverse*: given a measured readout `y_obs`, what `E_true` produced it, and with
what uncertainty?

We use a [Neyman construction](../docs/DECAL_pipeline.md):

1. The point estimate is `E_reco = f_med⁻¹(y_obs)` — invert the median curve
   (root-find with Brent's method).
2. The ±1σ interval comes from the **crossover**: invert the *lower* quantile
   curve for the upper energy bound, and the *upper* quantile curve for the lower
   bound. Intuitively, a measurement sitting on the low edge of the band is
   consistent with a higher true energy, and vice versa.

The panels below show this for all four readouts at three example true energies.
Where a readout's curves **flatten** (e.g. Raw Hits at high E, as pixels
saturate), the same `y_obs` maps to a much wider energy interval — exactly the
resolution loss the dashboard will quantify.

In [ ]:
from dashboard import invert_brent

e_grid_inv = np.linspace(1, 500, 1000)   # matches get_interpolators default range
# GIVEN: build the (low, med, high) interpolators for every readout. These are the
# forward curves you will invert.
interp = {name: get_interpolators(ens, xm_, ym_, device, e_grid_inv)
          for name, (ens, xm_, ym_) in ensembles.items()}

examples = [30.0, 150.0, 350.0]          # example true energies to invert

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, name in zip(axes.flat, ensembles):
    f_low, f_med, f_high = interp[name]
    eg = np.linspace(5, 400, 400)
    ax.plot(eg, f_med(eg), color=colors[name], lw=2, label=r"median $f_{med}$")
    ax.fill_between(eg, f_low(eg), f_high(eg), color=colors[name], alpha=0.18,
                    label=r"$\pm1\sigma$ quantiles")
    for e_true in examples:
        # TODO (your task): do the Neyman construction for this example energy.
        #   1. y_obs  = the median response at e_true  -> float(f_med(e_true))
        #   2. e_reco = invert_brent(y_obs, f_med)          # point estimate
        #   3. e_hi   = invert_brent(y_obs, f_low)          # CROSSOVER -> upper bound
        #   4. e_lo   = invert_brent(y_obs, f_high)         # CROSSOVER -> lower bound
        # Hint: the crossover is the whole trick — a measurement on the LOW edge of
        #   the band is consistent with a HIGHER true energy, so you invert f_low to
        #   get the upper energy bound (and f_high for the lower bound).
        # Then draw it: a dotted horizontal at y_obs, a shaded vertical span from
        #   e_lo to e_hi, and a marker at (e_reco, y_obs).
        # Expected: the dot sits on y=x of the median; the shaded interval is narrow
        #   where the curve is steep and WIDE where it flattens (e.g. Raw Hits high-E).
        # Solution: not distributed with this repository.
        raise NotImplementedError("fill this in")
    ax.set_title(f"{name}")
    ax.set_xlabel("$E_{true}$ [GeV]")
    ax.set_ylabel("readout value")
    ax.set_xlim(0, 400)
    ax.legend(loc="upper left", fontsize=9)
    ax.grid(True, alpha=0.3)
fig.suptitle("Neyman construction: a measured value (dotted) inverts to $E_{reco}$ (dot) "
             r"with a $\pm1\sigma$ energy interval (shaded)", fontsize=12)
plt.tight_layout()
plt.show()


## 7. Reconstruct over the full energy grid

Run the inversion across a dense grid of true energies for every readout,
collecting the response ratio (for the closure check above) and the ±1σ
resolution at each energy. The interpolators built in section 6 are reused.

In [ ]:
# TODO (your task): run the inversion across a dense energy grid for every readout.
# Hint: reco_metrics_over_grid(f_low, f_med, f_high) takes the three interpolators
#   for ONE readout and returns (e_true_grid, response_ratio, sigma_over_E). Build a
#   dict keyed by short names so the dashboard step can consume it:
#     reco = {
#         'Analog':  reco_metrics_over_grid(*interp['True Analog']),
#         'MIP':     reco_metrics_over_grid(*interp['MIP Proxy']),
#         'Hits':    reco_metrics_over_grid(*interp['Raw Hits']),
#         'Cluster': reco_metrics_over_grid(*interp['Naive 2D Clustering']),
#     }
# Expected: a `reco` dict with those 4 keys; each value is a 3-tuple of arrays.
#   The printout below should show sigma/E in the few-to-ten-percent range,
#   rising with energy.
# Solution: not distributed with this repository.
raise NotImplementedError("fill this in")

# Print headline numbers (works once `reco` exists)
et = reco["Analog"][0]
print("\n=== reconstructed resolutions (sigma_reco / E_true) ===")
for energy in (10, 100, 300):
    idx = np.argmin(np.abs(et - energy))
    print(f"  E={energy:>3d} GeV:  ", end="")
    for key in ("Analog", "MIP", "Hits", "Cluster"):
        print(f"{key.lower()}={reco[key][2][idx]:.4f}  ", end="")
    print()


## 8. How well does the inversion close?

Reconstruction should be **unbiased**: feeding the median response of a true
energy back through the inverse must return that same energy. The left panel
checks closure (`E_reco` vs `E_true`, should lie on `y = x`); the right panel is
the bias `E_reco/E_true − 1` in percent — flat at zero means a faithful
inversion.

This closure is *near-perfect by construction* (we invert the same median we
built), so it is a sanity check, not a resolution measurement — the real physics
is the band *width*, shown in the dashboard next. See
[handbook §13.1](../docs/handbook.md) for the caveat.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# TODO (your task): left panel = closure (E_reco vs E_true), right panel = bias.
# Hint: for each readout, unpack e_t, resp, res = reco[key]; E_reco is e_t * resp.
#   Left panel: plot E_reco vs e_t (one line per readout). Right panel: plot the
#   bias (resp - 1) in percent vs e_t.
# Expected: left = four lines hugging y=x; right = four lines flat near 0% (closure
#   is near-perfect by construction — it's a sanity check, not a resolution).
# Solution: not distributed with this repository.
raise NotImplementedError("fill this in")

axes[0].plot([0, 400], [0, 400], "k--", alpha=0.5, label="$y=x$ (perfect)")
axes[0].set_xlabel("$E_{true}$ [GeV]"); axes[0].set_ylabel("$E_{reco}$ [GeV]")
axes[0].set_title("Reconstruction closure"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].axhline(0, color="black", ls="--", alpha=0.5)
axes[1].set_ylim(-10, 10)
axes[1].set_xlabel("$E_{true}$ [GeV]"); axes[1].set_ylabel(r"bias $E_{reco}/E_{true}-1$ [%]")
axes[1].set_title("Reconstruction bias"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 9. The resolution dashboard

The payoff. Three panels: reconstructed linearity, resolution vs energy (the
saturation knee), and the stochastic term (σ/E vs 1/√E).

In [ ]:
# TODO (your task): draw the 3-panel physics dashboard from your `reco` dict.
# Hint: plot_dashboard(reco, out_path_prefix=None, show=True). That's the whole
#   call — the machinery is in analysis/dashboard.py.
# Expected: a 3-panel figure with all four readouts on each panel. Which curves
#   overlap? Does any curve change behaviour at high E -- on which panel is that
#   most visible?
#   SAVE this figure / your numbers — you'll compare it against the capstone
#   (improved clustering) and against the other particle.
# Solution: not distributed with this repository.
raise NotImplementedError("fill this in")


## What to look for in the dashboard

**Panel 1 — Linearity** (`E_reco / E_true`): should sit flat at 1.0. *This is partially trivial* (we're inverting the median surrogate with itself) — see [`docs/handbook.md`](../docs/handbook.md) §13.1 for caveats.

**Panel 2 — Resolution** (`σ_reco / E_true` vs `E_true`): the DECAL physics. Do any curves overlap — and why might that be? Does any curve diverge at high energy — at roughly what energy, and what does that say about the pixel pitch?

**Panel 3 — Stochastic** (`σ_reco / E_true` vs `1/√E_true`): straight line through origin = pure stochastic resolution `a/√E`. Deviations on the right side (low E) = sampling-fluctuation regime. Curving-up on the left (high E) = constant-term contributions from saturation/leakage.

For physics interpretation see [`docs/DECAL_pipeline.md`](../docs/DECAL_pipeline.md) §6 and [`docs/handbook.md`](../docs/handbook.md) §13.


---

## 10. CAPSTONE — improve the clustering, and *measure* it on the dashboard

Everything so far used the **baseline** clustering from nb02 — `naive_clusters()`, which counts **8-connected pixel groups per layer, independently**, and sums them. It is deliberately simple. The task is to **replace it with something better**, re-extract the cluster readout, retrain *just* the cluster ensemble, and **measure** whether the resolution improved.

### Why granularity matters here
At 100 µm pitch a shower is resolved into thousands of pixels across 30 layers. The naive baseline throws most of that structure away (it never looks across layers, never uses charge, never splits merged cores). A smarter clusterer that uses the 3-D / charge information should turn the readout into a tighter function of energy — i.e. a **lower σ/E** curve on the dashboard. If granularity didn't help, the curve wouldn't move. That is the experiment.

### Ideas to try (pick one; measure it)
- **3-D connected components across layers** instead of per-layer-independent (the most
  direct fix — a real shower is one object, not 30 disjoint slices).
- **Charge/energy weighting** — weight or threshold clusters by deposited charge so a
  faint single-pixel blip doesn't count the same as a dense core.
- **Density-based core splitting** (Molière-aware) — split merged cores using local
  pixel density so two overlapping sub-showers aren't merged into one.
- **A small learned clusterer** if you're ambitious.

**Success criterion:** a *lower* `σ/E` curve (and/or better linearity) for the Cluster
readout than the baseline, shown side-by-side on the dashboard, with the improvement
stated as a number (e.g. “σ/E at 100 GeV: X → Y, a Z% relative
change”).

Compare the photon and pion results: does the improved clusterer help one particle more than the other? Form a hypothesis about why before looking.

### 10.1 Write an improved clusterer

Implement `improved_clusters(...)` with the **same call signature and return type** as the baseline `naive_clusters(x, z, layer_idx, e)` from nb02 (it returns an integer cluster count for one event), so it can drop straight into the extraction loop. The baseline is reproduced below (read-only reference) so you can see exactly what you're beating.

In [ ]:
# GIVEN (read-only reference): the baseline you are trying to beat, copied from nb02.
# 8-connected components computed PER LAYER and summed — note it never looks across
# layers, never uses the charge `e` beyond the threshold mask.
CELL_SIZE  = 0.1            # mm — 100 um pixel pitch (same as nb02)
MIP_ENERGY = 85e-6          # GeV
THRESHOLD  = 0.5 * MIP_ENERGY

def naive_clusters(x, z, layer_idx, e):
    # 8-connected components per layer, summed (the BASELINE).
    m = e > THRESHOLD
    if not m.any():
        return 0
    xi = np.round(x[m] / CELL_SIZE).astype(np.int64)
    zi = np.round(z[m] / CELL_SIZE).astype(np.int64)
    li = layer_idx[m]
    total = 0
    for ly in np.unique(li):
        sel = li == ly
        cells_ = set(zip(xi[sel].tolist(), zi[sel].tolist()))
        seen = set()
        for cell in cells_:
            if cell in seen:
                continue
            # flood-fill an 8-connected component
            stack = [cell]; seen.add(cell)
            while stack:
                cx, cz = stack.pop()
                for dx in (-1, 0, 1):
                    for dz in (-1, 0, 1):
                        nb_ = (cx + dx, cz + dz)
                        if nb_ in cells_ and nb_ not in seen:
                            seen.add(nb_); stack.append(nb_)
            total += 1
    return total


In [ ]:
# TODO (your task): write improved_clusters(x, z, layer_idx, e) -> int.
#   SAME signature + return type as naive_clusters above, so it drops into the
#   extraction loop unchanged. Make it genuinely better than per-layer 8-connected.
# Hint (one concrete option — 3-D connected components across layers):
#   - keep the m = e > THRESHOLD mask
#   - build integer voxel coords (xi, zi, layer_idx[m]) — layer is the 3rd axis
#   - flood-fill in 3-D: a voxel's neighbours include +/-1 in x, z, AND layer
#   - count the connected components. (Optionally also weight/threshold by charge:
#     drop components whose summed e is below a few MIPs.)
#   scipy.ndimage.label on a sparse 3-D boolean grid is a fast alternative to a
#   hand-rolled flood-fill — either is fine.
# Expected: for a clean photon shower, FEWER, larger clusters than the baseline
#   (one shower core instead of ~30 per-layer fragments); the count becomes a
#   tighter function of E_true. Sanity-check on one event: improved_clusters(...)
#   should return a small-ish integer (order 1-50), not thousands.
# Solution: none exists — this is the open R&D part of the project.
raise NotImplementedError("fill this in")


### 10.2 Re-extract the cluster readout with your improved clusterer

Re-run the **cluster readout only** over your particle's `.root` files using `improved_clusters()` in place of `naive_clusters()`, producing a new per-event array `all_cluster_improved` aligned with `all_truth`. The simplest path is to copy the extraction loop from nb02 and swap the clustering call — the other three readouts don't change, so you only need to recompute clusters. (Keep the same `+y` wedge selection, `CELL_SIZE`, and `THRESHOLD` so it's an apples-to-apples comparison.)

In [ ]:
import glob
DATA_BASE = os.environ.get("CALOMAPS_DATA_BASE", os.path.expanduser("~/CALOMAPS-data"))

# TODO (your task): produce all_cluster_improved, one value per event, aligned with
#   all_truth (same events, same order, same wedge selection as nb02).
# Hint: reuse the nb02 extraction harness — the per-file reader, the +y wedge cut,
#   the (x, z, layer_idx, e) you already build there — but call improved_clusters()
#   instead of naive_clusters(). You can re-run the FULL nb02 loop with the swap, or
#   (faster) recompute ONLY the cluster readout per event. The file glob is FILE_GLOB
#   under os.path.join(DATA_BASE, DATASET).
# Hint: this can be slow — the same ProcessPoolExecutor pattern from nb02 helps.
# Expected: all_cluster_improved.shape == all_truth.shape; on average FEWER clusters
#   per event than all_cluster (you merged per-layer fragments), and a TIGHTER scatter
#   vs E_true — eyeball it against the baseline before retraining.
# Solution: adapt the nb02 extraction loop above; the clusterer itself is yours.
raise NotImplementedError("fill this in")

# Quick before/after sanity scatter (works once all_cluster_improved exists):
# fig, ax = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
# ax[0].scatter(x_train, all_cluster[valid],          s=4, alpha=0.15, color='darkorchid'); ax[0].set_title('baseline clusters')
# ax[1].scatter(x_train, all_cluster_improved[valid], s=4, alpha=0.15, color='teal');       ax[1].set_title('improved clusters')
# for a in ax: a.set_xlabel('$E_{true}$ [GeV]'); a.grid(True, alpha=0.3)
# ax[0].set_ylabel('cluster count'); plt.tight_layout(); plt.show()


### 10.3 Retrain *only* the cluster ensemble on the improved readout

Train one new ensemble on `(x_train, all_cluster_improved[valid])` and save it next to the others (e.g. `ens_cluster_improved.pth`). The analog/MIP/hits ensembles are unchanged — you're isolating the effect of the clustering algorithm.

In [ ]:
# TODO (your task): train + load the improved-cluster ensemble.
# Hint: mirror the training call from section 3 —
#   ens, xmax, ymax = train_one_ensemble(x_train, all_cluster_improved[valid], device,
#                                        name='Improved Clustering', seed_base=5000)
#   save_ensemble(ens, xmax, ymax, os.path.join(ENSEMBLE_DIR, 'ens_cluster_improved.pth'))
#   ens_ci, xci, yci = load_ensemble(os.path.join(ENSEMBLE_DIR, 'ens_cluster_improved.pth'), device)
# Expected: one ensemble, ~20 models, on device=cuda; loss curve converges like the
#   others. Use a NEW seed_base so it's an independent bootstrap.
# Solution: same machinery as section 3; only the input array changed.
raise NotImplementedError("fill this in")


### 10.4 Quantify the payoff on the dashboard

Build the interpolators + reco metrics for the **improved** cluster ensemble exactly as in sections 6–7, then plot the dashboard with **both** cluster readouts so the curves sit on top of each other. Finally, print the relative improvement in `σ/E` at a few energies. This number — and the gap between the two Cluster curves on panel 2 — is your capstone result.

In [ ]:
# TODO (your task): compare baseline vs improved clustering on the dashboard.
# Hint:
#   1. f_lo, f_med, f_hi = get_interpolators(ens_ci, xci, yci, device, e_grid_inv)
#   2. reco_cmp = {
#          'Cluster (baseline)': reco['Cluster'],
#          'Cluster (improved)': reco_metrics_over_grid(f_lo, f_med, f_hi),
#      }
#      plot_dashboard(reco_cmp, out_path_prefix=None, show=True)
#   3. quantify: for E in (10, 100, 300):
#          idx_b = np.argmin(np.abs(reco['Cluster'][0] - E))
#          base = reco['Cluster'][2][idx_b]
#          imp  = reco_cmp['Cluster (improved)'][2][idx_b]
#          print(f'E={E:>3} GeV  sigma/E  baseline={base:.4f}  improved={imp:.4f}  '
#                f'rel.improvement={100*(base-imp)/base:+.1f}%')
# Expected: two Cluster curves on panel 2 that can be compared quantitatively.
#   Whether the improved curve sits below the baseline -- and where -- is the result.
#   A curve that does not improve is also a reportable result; document the reason.
# Solution: none exists — this is the open part of the project.
raise NotImplementedError("fill this in")


### Capstone wrap-up — photon vs pion

Two results come out of this notebook:

1. **Readout ranking** (sections 1–9): which of the four *baseline* readouts reconstructs energy best, and where each one breaks.
2. **The granularity payoff** (this capstone): how much a *smarter* clusterer that uses the 100 µm 3-D structure tightens the resolution over the naive baseline — stated as a number.

Put the two dashboards side by side: **photon vs pion** is the headline contrast — describe it in numbers. Write down the three numbers — best baseline σ/E, improved-cluster σ/E, and the relative gain — for both particles. That table summarizes the result.

## Suggested follow-up experiments

1. **Pixel pitch scan**: re-run the entire pipeline at 25, 50, 200 µm pixel pitch (edit `ECal_cell_size` in `geometry/SiD_TestBeam.xml`, regenerate sim, re-extract, retrain). Plot how the saturation knee moves.
2. **Particle type**: gamma and pi+ are wired into the notebooks via `PARTICLE`; try `CALOMAPS_GUN_PARTICLE=proton bash $CALOMAPS_HOME/sim/generate_batched.sh` (no file edits) and extend the `PARTICLE` switch in nb01-03 for the new dataset. How does the resolution change?
3. **More models per ensemble**: edit `num_models=20` in `analysis/train_ensembles.py`. Does the resolution prediction tighten with more bootstrap samples?
4. **Held-out test set**: the current linearity panel is trivially perfect because we use the same curve for forward and inverse. Modify the pipeline to bootstrap-resample real events and reconstruct each one — what does linearity look like then?
